In [1]:
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path
from sklearn.model_selection import train_test_split

# ── Paths ─────────────────────────────────────────────
CORPUS_DIR = Path("/kaggle/input/datasets/troymerales/bisaya-audio")
SPLIT_DIR = Path("/kaggle/working/bisaya_split")

TRAIN_DIR = SPLIT_DIR / "train"
TEST_DIR = SPLIT_DIR / "test"
TRAIN_DIR.mkdir(parents=True, exist_ok=True)
TEST_DIR.mkdir(parents=True, exist_ok=True)

# ── Find all Parquet shards ────────────────────────────
parquet_files = sorted(CORPUS_DIR.glob("*.parquet"))

print(f"Found {len(parquet_files)} Parquet files")

# ── Pass 1: collect every speaker_id, one shard at a time ─────────────
# Reading only the speaker_id column keeps this cheap regardless of how
# large the audio column is.
all_speakers = set()
total_rows = 0
for path in parquet_files:
    ids = pq.read_table(path, columns=["speaker_id"]).column("speaker_id").to_pylist()
    all_speakers.update(ids)
    total_rows += len(ids)

speakers = sorted(all_speakers)
print(f"\nTotal utterances: {total_rows:,}")
print(f"Total speakers: {len(speakers):,}")

# ── Speaker-independent 90/10 split ──────────────────
train_speakers, test_speakers = train_test_split(
    speakers,
    test_size=0.10,
    random_state=42
)

train_speakers = set(train_speakers)
test_speakers = set(test_speakers)

# ── Pass 2: filter and write each shard independently ──────────────────
# Deliberately NOT concatenated into one big train.parquet/test.parquet
# first: pa.concat_tables() across shards followed by a single write_table()
# produces a multi-row-group file, and reading that back (even just with
# pq.read_table(), before any pandas/to_pylist conversion) raises
# "ArrowNotImplementedError: Nested data conversions not implemented for
# chunked array outputs" on this pyarrow build's Dataset scanner for nested
# (audio struct) columns spread across >1 row group. Forcing everything into
# a single row group instead (e.g. via combine_chunks()) just trades that
# for "ArrowInvalid: offset overflow while concatenating arrays", since the
# combined audio-bytes data exceeds the 2 GB cap on 32-bit binary offsets.
# Writing one output file per source shard sidesteps both: each shard was
# already a single row group on disk (that's why the original per-shard
# loading loop above works fine), and filtering a single-chunk table keeps
# it a single chunk, so nothing here ever needs a multi-row-group merge.
train_rows = test_rows = 0
train_speakers_seen, test_speakers_seen = set(), set()

for i, path in enumerate(parquet_files):
    print(f"Splitting: {path.name}")
    table = pq.read_table(path)
    speaker_values = table.column("speaker_id").to_pylist()

    train_mask = pa.array([s in train_speakers for s in speaker_values])
    test_mask = pa.array([s in test_speakers for s in speaker_values])

    train_part = table.filter(train_mask)
    test_part = table.filter(test_mask)

    if train_part.num_rows:
        pq.write_table(train_part, TRAIN_DIR / f"train-{i:05d}.parquet")
        train_rows += train_part.num_rows
        train_speakers_seen.update(s for s in speaker_values if s in train_speakers)
    if test_part.num_rows:
        pq.write_table(test_part, TEST_DIR / f"test-{i:05d}.parquet")
        test_rows += test_part.num_rows
        test_speakers_seen.update(s for s in speaker_values if s in test_speakers)

# ── Results ────────────────────────────────────────────
print("\n── Split ──")
print(f"Train: {train_rows:,} utterances, {len(train_speakers_seen):,} speakers")
print(f"Test:  {test_rows:,} utterances, {len(test_speakers_seen):,} speakers")
print(f"\nSaved to: {SPLIT_DIR} (train/*.parquet, test/*.parquet)")


KeyboardInterrupt



In [ ]:
from pathlib import Path

IS_KAGGLE = Path("/kaggle").exists()

# ── Original source corpus ────────────────────────────
CORPUS_DIR = (
    Path("/kaggle/input/datasets/troymerales/bisaya-audio")
    if IS_KAGGLE
    else Path("data/bisaya_audio").resolve()
)

# ── Train/test split ──────────────────────────────────
SPLIT_DIR = (
    Path("/kaggle/working/bisaya_split")
    if IS_KAGGLE
    else Path("./bisaya_split").resolve()
)

# Each of these holds one Parquet file per source shard (see the split
# cell above) -- not a single train.parquet/test.parquet file.
TRAIN_DIR = SPLIT_DIR / "train"
TEST_DIR = SPLIT_DIR / "test"

# ── ASR working directory ─────────────────────────────
WORK_DIR = (
    Path("/kaggle/working/bisaya_asr")
    if IS_KAGGLE
    else Path("./bisaya_asr").resolve()
)

WORK_DIR.mkdir(parents=True, exist_ok=True)

KALDI_ROOT = WORK_DIR / "kaldi"
DATA_ROOT = WORK_DIR / "data"
MFCC_ROOT = WORK_DIR / "mfcc"
EXP_ROOT = WORK_DIR / "exp"

CHECKPOINT_PATH = (
    Path("/kaggle/working/checkpoint.tar.gz")
    if IS_KAGGLE
    else WORK_DIR.parent / "checkpoint.tar.gz"
)

print("IS_KAGGLE:", IS_KAGGLE)
print("CORPUS:", CORPUS_DIR)
print("TRAIN_DIR:", TRAIN_DIR)
print("TEST_DIR:", TEST_DIR)
print("WORK:", WORK_DIR)

In [ ]:
def sh(cmd, cwd=None, check=True, env=None):
    # Streams output live -- used for every Kaldi binary/script invocation
    # below instead of `!` magics, so it behaves the same interactively or
    # when a committed Kaggle version replays the notebook top to bottom.
    print(f"$ {cmd}")
    proc = subprocess.run(cmd, shell=True, cwd=cwd, check=check, executable="/bin/bash", env=env)
    return proc.returncode


def save_checkpoint():
    print(f"Archiving {WORK_DIR} -> {CHECKPOINT_PATH} ...")
    with tarfile.open(CHECKPOINT_PATH, "w:gz") as tar:
        tar.add(WORK_DIR, arcname=WORK_DIR.name)
    size_mb = CHECKPOINT_PATH.stat().st_size / 1e6
    print(f"Done. {size_mb:.1f} MB.")


def restore_checkpoint(archive_path=None):
    archive_path = Path(archive_path) if archive_path else CHECKPOINT_PATH
    print(f"Restoring {archive_path} -> {WORK_DIR.parent} ...")
    with tarfile.open(archive_path, "r:gz") as tar:
        tar.extractall(WORK_DIR.parent)
    print("Done. Re-run the setup/config cells above, then continue -- completed "
          "GMM stages and nnet3 iterations will be detected and skipped automatically.")


# --- Optional: push checkpoints to a personal Kaggle Dataset mid-session ---
# Only needed if mid-session crashes (not just session-end timeouts) are a
# real concern for you; requires the Kaggle API configured (kaggle.json).
#
# KAGGLE_DATASET_SLUG = "your-username/asr-train-checkpoint"
# def push_checkpoint_to_dataset():
#     save_checkpoint()
#     sh(f"kaggle datasets version -p {CHECKPOINT_PATH.parent} -m 'checkpoint update' "
#        f"-d {KAGGLE_DATASET_SLUG}")

In [ ]:
import subprocess

sh("apt-get update -qq && apt-get install -y -qq "
   "build-essential automake autoconf libtool subversion git zlib1g-dev "
   "gfortran libatlas-base-dev sox")

sh("pip install -q tqdm")

In [ ]:
if not KALDI_ROOT.exists():
    sh(f"git clone --depth 1 https://github.com/kaldi-asr/kaldi.git {KALDI_ROOT}")
else:
    print(f"{KALDI_ROOT} already exists -- skipping clone (resumed from checkpoint).")

# tools/: OpenFST and other bundled dependencies. Incremental -- safe to
# re-run after an interruption, make only redoes unfinished work.
sh("make -j$(nproc)", cwd=KALDI_ROOT / "tools")

# KenLM: Kaldi's bundled installer builds lmplz/build_binary without needing
# SRILM's license-gated download.
sh("extras/install_kenlm.sh", cwd=KALDI_ROOT / "tools")

In [ ]:
src_dir = KALDI_ROOT / "src"

# use-cuda=no: this targets CPU-only environments per the earlier hardware
# discussion (no CUDA-capable GPU locally). If your Kaggle session has a GPU
# attached, drop --use-cuda=no to let nnet3 training use it.
if not (src_dir / "kaldi.mk").exists():
    sh("./configure --shared --use-cuda=no", cwd=src_dir)

sh("make -j$(nproc) depend", cwd=src_dir)
sh("make -j$(nproc)", cwd=src_dir)

print("Kaldi build complete." if (src_dir / "bin" / "compute-mfcc-feats").exists()
      else "WARNING: expected binary not found -- check the build log above for errors.")

In [ ]:
path_sh_lines = [
    f"export KALDI_ROOT={KALDI_ROOT}",
    "export PATH=$PWD/utils/:$KALDI_ROOT/tools/openfst/bin:$PWD:$PATH",
    ". $KALDI_ROOT/tools/config/common_path.sh",
    "export LC_ALL=C",
]
(WORK_DIR / "path.sh").write_text("\n".join(path_sh_lines) + "\n")

cmd_sh_lines = [
    "export train_cmd=run.pl",
    "export decode_cmd=run.pl",
    "export mkgraph_cmd=run.pl",
]
(WORK_DIR / "cmd.sh").write_text("\n".join(cmd_sh_lines) + "\n")

wsj_s5 = KALDI_ROOT / "egs" / "wsj" / "s5"
for name in ("steps", "utils"):
    link = WORK_DIR / name
    if not link.exists():
        link.symlink_to(wsj_s5 / name)

print("path.sh, cmd.sh, steps/, utils/ ready under", WORK_DIR)

In [ ]:
import pandas as pd
import pyarrow.parquet as pq

def load_parquet_split(corpus_dir, glob_pattern="*.parquet"):
    files = sorted(Path(corpus_dir).glob(glob_pattern))
    if not files:
        return pd.DataFrame()
    dfs = []
    for file in files:
        # Each shard file here has a single row group (written that way by
        # the split cell above), so a plain read is safe -- but to_pylist()
        # instead of to_pandas() is kept anyway since it's just as simple
        # and avoids ever depending on that invariant holding.
        rows = pq.read_table(file).to_pylist()
        for row in rows:
            row["source_file"] = file.name
        dfs.append(pd.DataFrame(rows))
    return pd.concat(dfs, ignore_index=True)


train_files = sorted(TRAIN_DIR.glob("*.parquet"))
test_files = sorted(TEST_DIR.glob("*.parquet"))
print(f"Train shard files: {len(train_files)} under {TRAIN_DIR}")
print(f"Test shard files:  {len(test_files)} under {TEST_DIR}")

if not train_files:
    print("\nWARNING: no train shard files found. Run the split cell above "
          "first (it writes TRAIN_DIR/*.parquet and TEST_DIR/*.parquet).")

train_df = load_parquet_split(TRAIN_DIR)
test_df = load_parquet_split(TEST_DIR)

print(f"\ntrain_df: {len(train_df)} rows, {train_df['speaker_id'].nunique() if len(train_df) else 0} speakers")
print(f"test_df:  {len(test_df)} rows, {test_df['speaker_id'].nunique() if len(test_df) else 0} speakers")